# 02 - From a STAC catalog to data

### Learning objectives

- Get familiar with the catalog usage
- Improve data interoperability with public data repositories
- Learn how to:
  - Download the catalog from a data repo
  - Visualize the catalog
  - Inspect and filter the data based on catalog
  - Download the selected data

### Workflow

1. Find a public data repo of interest;
2. Download catalog from the repo;
3. Load the catalog and inspect the metadata;
4. Filter dataset;
5. Retrieve download links;
6. Download the data. 

## Expected input and output

### Input folder

The notebook expects a dataset from a public data repository (e.g., Zenodo), which contains STAC-based catalogs.

### Output folder
```
data/
├── catalog/
│   ├── catalog.json
│   └── collections/
│       ├── collection.json
│       ├── epoch_1.json
│       ├── epoch_2.json
│       └── ...
└── download/
    └── dataset/
        ├── epoch_1.laz
        ├── epoch_2.laz
        └── ...
```

## 1. Define the data repository

In [1]:
# installation
# %pip install -q pystac ipyleaflet ipywidgets

In [1]:
from pathlib import Path
from urllib.parse import urlsplit, unquote
from urllib.request import urlretrieve

import pystac
import ipywidgets as widgets
from ipyleaflet import Map, GeoJSON
from IPython.display import display

In [2]:
# Define the data directories
DATA = Path("data")
CATALOG_DIR = DATA / "catalog"
DOWNLOAD_DIR = DATA / "download" / "dataset"
CATALOG_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# define the calog URL
CATALOG_URL = "https://zenodo.org/records/21460578/files/catalog.zip"
# CATALOG_URL = "https://zenodo.org/records/21460578/files/catalog.json"
# CATALOG_URL = r"C:\Users\jiapan\00_tmp\2026_Isar_data_pub\catalog\catalog.json"

print("Source:", CATALOG_URL)

Source: https://zenodo.org/records/21460578/files/catalog.zip


## 2. Download the catalog

In [4]:
from zipfile import ZipFile
import shutil

source = CATALOG_URL

if source.lower().split("?")[0].endswith(".zip"):
    archive = DATA / "catalog.zip"
    urlretrieve(source, archive)
    extracted = DATA / "catalog_extracted"
    with ZipFile(archive) as z:
        z.extractall(extracted)
    source = next(extracted.rglob("catalog.json"))

remote = pystac.Catalog.from_file(str(source))
remote.fully_resolve()

# For .laz files uploaded separately to the same Zenodo record.
if "zenodo.org/records/" in CATALOG_URL:
    record_url = CATALOG_URL.split("/files/")[0]
    for item in remote.get_items(recursive=True):
        for asset in item.assets.values():
            if not urlsplit(asset.href).scheme and asset.href.lower().endswith(".laz"):
                asset.href = f"{record_url}/files/{Path(asset.href).name}?download=1"

remote.make_all_asset_hrefs_absolute()
remote.normalize_and_save(
    str(CATALOG_DIR.resolve()),
    catalog_type=pystac.CatalogType.SELF_CONTAINED,
)

if "extracted" in locals():
    shutil.rmtree(extracted)
    
print("Catalog saved:", CATALOG_DIR / "catalog.json")

Catalog saved: data\catalog\catalog.json


## 3. Load the catalog and inspect the metadata

In [5]:
catalog = pystac.Catalog.from_file(str(CATALOG_DIR / "catalog.json"))
collections = list(catalog.get_all_collections())
items = list(catalog.get_items(recursive=True))

print("Catalog:", catalog.id, "—", catalog.title)
print("Collections:", len(collections), "| Items (epochs):", len(items))
for collection in collections:
    print("Collection:", collection.id, "|", collection.title)
    print("  Bboxes:", collection.extent.spatial.bboxes)
    print("  Time intervals:", collection.extent.temporal.intervals)

for item in items[:5]:
    print(item.id, "|", item.datetime, "| assets:", list(item.assets))
if items:
    print("First item properties:", items[0].properties)


Catalog: isar-catalog — Isar topo4d STAC Catalog
Collections: 1 | Items (epochs): 6
Collection: isar-pointclouds | Isar multi-temporal point clouds
  Bboxes: [[11.304060903950353, 47.525938711635945, 11.31717859810396, 47.53282626569896]]
  Time intervals: [[datetime.datetime(2024, 8, 12, 13, 26, 52, tzinfo=tzutc()), datetime.datetime(2026, 9, 23, 9, 34, 19, tzinfo=tzutc())]]
Isar_ULS_20250325 | 2025-03-25 12:15:51+00:00 | assets: ['pointcloud']
Isar_ULS_20250909 | 2025-09-09 09:14:44+00:00 | assets: ['pointcloud']
Isar_ULS_20260324 | 2026-03-24 09:28:58+00:00 | assets: ['pointcloud']
Isar_ULS_20260923 | 2026-09-23 09:34:19+00:00 | assets: ['pointcloud']
Isar_UPH_20240812 | 2024-08-12 13:26:52+00:00 | assets: ['pointcloud']
First item properties: {'proj:code': 'EPSG:32632', 'pc:count': 348536329, 'pc:type': 'lidar', 'instruments': ['DJI Zenmuse L2'], 'topo4d:timezone': 'UTC+1', 'topo4d:data_type': 'pointcloud', 'topo4d:acquisition_mode': 'ULS', 'topo4d:orientation': 'Nadir', 'topo4d:sp

## 4. Filter dataset

In [6]:
# Optional spatial filter, e.g. (-78.0, 39.0, -75.0, 42.0)
AOI = None

def bbox2d(item):
    b = item.bbox
    if b is None:
        return None
    return (b[0], b[1], b[3], b[4]) if len(b) == 6 else tuple(b)

def overlaps(a, b):
    return a[0] <= b[2] and a[2] >= b[0] and a[1] <= b[3] and a[3] >= b[1]

def item_time(item):
    return item.datetime or item.common_metadata.start_datetime

def footprint(item):
    if item.geometry:
        return item.geometry
    b = bbox2d(item)
    if b is None:
        return None
    w, s, e, n = b
    return {"type": "Polygon", "coordinates": [[[w,s],[e,s],[e,n],[w,n],[w,s]]]}

dates = sorted({item_time(i).date().isoformat() for i in items if item_time(i)})
assert dates, "No timestamps were found in this STAC catalog."

bounds = [bbox2d(i) for i in items if bbox2d(i)]
assert bounds, "No item bounding boxes were found."
west, south = min(b[0] for b in bounds), min(b[1] for b in bounds)
east, north = max(b[2] for b in bounds), max(b[3] for b in bounds)


In [11]:
date_slider = widgets.SelectionRangeSlider(
    options=dates,
    index=(0, len(dates) - 1),
    description="Dates",
    continuous_update=False,
    readout=False,
    layout=widgets.Layout(width="650px"),
)
m = Map(center=((south + north) / 2, (west + east) / 2), zoom=5,
        scroll_wheel_zoom=True, layout=widgets.Layout(height="430px"))
m.fit_bounds([[south, west], [north, east]])
layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#126b88", "weight": 2, "fillOpacity": 0.18},
    hover_style={"color": "#e08023", "fillOpacity": 0.35},
)
m.add(layer)
status = widgets.HTML()
selected_items = []

def update_map(change=None):
    global selected_items
    first, last = date_slider.value
    selected_items = [i for i in items
                      if item_time(i) and first <= item_time(i).date().isoformat() <= last
                      and (AOI is None or (bbox2d(i) and overlaps(bbox2d(i), AOI)))]
    layer.data = {
        "type": "FeatureCollection",
        "features": [
            {"type": "Feature", "geometry": footprint(i),
             "properties": {"id": i.id, "datetime": item_time(i).isoformat()}}
            for i in selected_items if footprint(i)
        ],
    }
    status.value = (
        f"<b>{len(selected_items)} of {len(items)} items selected</b>"
        f" | {first} → {last}"
    )

date_slider.observe(update_map, names="value")
update_map()
display(widgets.VBox([date_slider, status, m]))


## 5. Retrieve download links

In [ ]:
ASSET_SUFFIX = ".laz" 

links = [
    (item.id, key, asset.get_absolute_href())
    for item in selected_items
    for key, asset in item.assets.items()
    if (asset.get_absolute_href() or "").split("?", 1)[0].lower().endswith(ASSET_SUFFIX)
]

print(f"Found {len(links)} matching assets")
for item_id, key, href in links[:10]:
    print(f"{item_id} | {key} | {href}")


Found 2 matching assets
Isar_UPH_20240812 | pointcloud | https://zenodo.org/records/21460578/files/Isar_UPH_20240812.copc.laz
Isar_UPH_20241105 | pointcloud | https://zenodo.org/records/21460578/files/Isar_UPH_20241105.copc.laz


## 6. Download the data

In [18]:
from IPython.display import display
import ipywidgets as widgets

for item_id, key, href in links[:2]:
    filename = Path(unquote(urlsplit(href).path)).name
    target = DOWNLOAD_DIR / filename
    temp = target.with_suffix(target.suffix + ".part")

    if target.exists():
        print("Skipped (already exists):", filename)
        continue

    print("Downloading:", filename)

    progress = widgets.IntProgress(
        value=0, min=0, max=100,
        bar_style="info",
        layout=widgets.Layout(width="400px"),
    )
    percent_label = widgets.Label(value="0%")

    display(widgets.HBox([progress, percent_label]))

    def reporthook(block_num, block_size, total_size):
        if total_size > 0:
            downloaded = min(block_num * block_size, total_size)
            percent = int(downloaded / total_size * 100)

            progress.value = percent
            percent_label.value = f"{percent}%"

    try:
        urlretrieve(href, temp, reporthook=reporthook)
        temp.replace(target)

        progress.value = 100
        percent_label.value = "100%"
        progress.bar_style = "success"
    finally:
        temp.unlink(missing_ok=True)

    print(f"Saved: {target} ({target.stat().st_size / 1e6:.1f} MB)")

Downloading: Isar_UPH_20240812.copc.laz


Saved: data\download\dataset\Isar_UPH_20240812.copc.laz (756.1 MB)
Downloading: Isar_UPH_20241105.copc.laz


Saved: data\download\dataset\Isar_UPH_20241105.copc.laz (955.9 MB)


## Next steps

Congrats, if you manage to download the selected point cloud files. Now you can move on to the next notebook:

- **NB3 — Catalog for Analysis**: run a simple analysis such as M3C2 and visualize the result.
